# Test-set error analysis from a per-point dump

Dump produced by `scripts/dump_point_predictions.py` (one row per annotated point). All metric helpers
use the same definitions as `scripts/evaluate.py`, so `summary_by(df)` equals the pooled block of
`metrics.json` and `summary_by(df, "source_id")` equals `per_unit.csv`. Filter `df` first to restrict any
analysis (e.g. `df[df.concept_mapped]`, one source, one label).

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, "..")
from mermaidseg.evaluation.point_dump import (
    PointDump, summary_by, class_metrics, taxon_metrics, binary_metrics, confusions,
)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)

DUMP = "../eval_out/point_dumps/scratch100k_base/test"  # <- set
d = PointDump.load(DUMP)
df = d.df
print(d.meta["checkpoint"], d.meta["split"], len(df), "points,", df.source_id.nunique(), "sources")
print("ranks:", d.ranks)

## 1. Pooled vs macro (per-source mean)
A large gap means a few big sources dominate the pooled number.

In [ ]:
per_src = summary_by(df, "source_id")
metric_cols = [c for c in per_src.columns if c.endswith(("accuracy", "acc_all", "acc_living", "miou", "f1"))]
pd.DataFrame({"pooled": summary_by(df)[metric_cols].iloc[0],
              "macro_mean": per_src[metric_cols].mean(),
              "macro_median": per_src[metric_cols].median()}).round(3)

## 2. Per source
Sorted by points; look for big sources with low accuracy.

In [ ]:
cols = ["source_id", "num_points", "class_points_scored", "class_accuracy", "class_miou"] + \
       [f"{r}_acc_all" for r in d.ranks] + ["binary_macro_f1"]
per_src[cols].sort_values("num_points", ascending=False).round(3).head(40)

In [ ]:
# Share of the pooled class-accuracy error contributed by each source
scored = df[df.gt_mermaid_id != 0]
err = (scored.gt_mermaid_id != scored.pred_mermaid_id).groupby(scored.source_id).sum()
(err / err.sum()).sort_values(ascending=False).cumsum().head(15).round(3)

## 3. Per MERMAID class (class head)

In [ ]:
class_metrics(df).sort_values("n_gt", ascending=False).round(3).head(40)

In [ ]:
confusions(df, "mermaid").head(25).round(3)

## 4. Per taxon at each rank (concept head)

In [ ]:
rank = "genus"
taxon_metrics(df, rank).sort_values("n_gt", ascending=False).round(3).head(30)

In [ ]:
confusions(df, rank).head(25).round(3)

## 5. Binary concepts

In [ ]:
binary_metrics(df).sort_values("n_valid", ascending=False).round(3)

## 6. Concept x source
Per-source F1 of one binary concept / recall of one taxon. Pivot to compare sources.

In [ ]:
bm = binary_metrics(df, "source_id")
bm.pivot(index="source_id", columns="concept", values="f1").round(2).head(30)

In [ ]:
tm = taxon_metrics(df, "genus", "source_id")
tm[tm.n_gt >= 50].pivot(index="taxon", columns="source_id", values="recall").round(2)

## 7. Per raw CoralNet label
Each source uses its own label set; a label that is consistently "wrong" across many points is often
a mapping / taxonomy issue rather than a model issue. Compare what the GT says vs what the model predicts.

In [ ]:
g = df.groupby("source_label_name", observed=True)
lab = pd.DataFrame({
    "n": g.size(),
    "n_sources": g.source_id.nunique(),
    "gt_mermaid": g.gt_mermaid.agg(lambda s: s.mode().iloc[0] if s.notna().any() else None),
    "top_pred_mermaid": g.pred_mermaid.agg(lambda s: s.mode().iloc[0]),
    "mermaid_acc": g.apply(lambda x: (x.gt_mermaid_id == x.pred_mermaid_id)[x.gt_mermaid_id != 0].mean()),
    "finest_rank": g.gt_finest_rank.agg(lambda s: s.mode().iloc[0] if s.notna().any() else None),
    "concept_mapped": g.concept_mapped.first(),
})
lab.sort_values("n", ascending=False).round(3).head(40)

In [ ]:
# Labels with no MERMAID class / no concept row (dropped from class / concept metrics)
print("unscored for class head:", (df.gt_mermaid_id == 0).mean().round(3), "of points")
print("missing from concept table:", (~df.concept_mapped).mean().round(3), "of points")
lab[lab.gt_mermaid.isna()].sort_values("n", ascending=False).head(20)

## 8. Confidence and position
Confident errors point at label noise / mapping; low-confidence errors at genuine ambiguity.

In [ ]:
s = scored.assign(correct=scored.gt_mermaid_id == scored.pred_mermaid_id,
                 conf_bin=pd.cut(scored.pred_mermaid_prob, [0, .3, .5, .7, .9, .99, 1]))
s.groupby("conf_bin", observed=True).agg(n=("correct", "size"), acc=("correct", "mean")).round(3)

In [ ]:
s.assign(edge_bin=pd.cut(s.edge_dist, [0, .01, .03, .1, .5], include_lowest=True)) \
 .groupby("edge_bin", observed=True).agg(n=("correct", "size"), acc=("correct", "mean")).round(3)

## 9. Per-point probabilities
`d.rank_probs(rank, subset)` gives the full distribution over a rank's taxa (from `concept_probs.npy`).

In [ ]:
sub = df[df.gt_genus.astype(str) == "acropora"].head(2000)
d.rank_probs("genus", sub).mean().sort_values(ascending=False).head(10).round(3)